# IOC Olive Oil Market - Exploration and Cleaning

**Module Project 1: Data Storytelling**  
**Alberto Gomez Soteres & Burak Donbekci**

This notebook explores and cleans International Olive Council (IOC) olive oil data for Spain and Türkiye.

FAOSTAT covers olive trees. IOC covers the oil itself: production, consumption, exports, and imports by crop year. The goal is to understand the dataset, check its quality, and prepare clean data for the next stages of the project.

## 1. Imports

First, we import the tools needed to load, explore, and clean the data.

In [1]:
from pathlib import Path
import sys

import pandas as pd

## 2. Load the Data

Next, we load the raw IOC dashboard extract from the project's data folder. The original download uses thousand tonnes and a crop year such as 1990/91, which runs from October to September.

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

file_path = project_root / "data" / "raw" / "ioc_olive_oil.csv"

df = pd.read_csv(file_path)

df.head()

,Haverst period,Country,Product Type,Indicator,Tonnes
0,1990/91,Algeria,OO,C,7.0
1,1990/91,Algeria,OO,E,0.0
2,1990/91,Algeria,OO,I,0.0
3,1990/91,Algeria,OO,P,6.0
4,1990/91,Argentina,OO,C,4.0


## 3. Understand the Dataset Structure

Before cleaning the data, we check its size, columns, and data types. This helps us understand how the IOC dashboard file is organized.

In [3]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 9886
Columns: 5


In [4]:
df.columns.tolist()

['Haverst period', 'Country', 'Product Type', 'Indicator', 'Tonnes']

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9886 entries, 0 to 9885
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Haverst period  9886 non-null   object 
 1   Country         9886 non-null   object 
 2   Product Type    9886 non-null   object 
 3   Indicator       9886 non-null   object 
 4   Tonnes          9886 non-null   float64
dtypes: float64(1), object(4)
memory usage: 386.3+ KB


The first column is spelled `Haverst period` in the original IOC download. We keep that spelling in the raw file and fix it later during cleaning.

## 4. Dataset Coverage

Next, we check the scope of the dataset. We verify the product types, indicators, countries, and time period, and confirm that Spain and Türkiye are both included.

In [6]:
print("Product types:")
print(df["Product Type"].value_counts())
print("\nIndicators:")
print(df["Indicator"].value_counts())

Product types:
Product Type
OO    5331
TO    4555
Name: count, dtype: int64

Indicators:
Indicator
C    2853
I    2851
E    2697
P    1485
Name: count, dtype: int64


`OO` is olive oil and `TO` is table olives. `P`, `C`, `E`, and `I` are production, consumption, exports, and imports. For this project we keep olive oil only.

In [7]:
olive_oil = df[df["Product Type"] == "OO"].copy()

print("Olive oil rows:", olive_oil.shape[0])
print("Number of countries:", olive_oil["Country"].nunique())

for country in ["Spain", "Türkiye"]:
    print(country, "->", country in olive_oil["Country"].values)

Olive oil rows: 5331
Number of countries: 66
Spain -> True
Türkiye -> True


In [8]:
print("First crop year:", olive_oil["Haverst period"].min())
print("Last crop year:", olive_oil["Haverst period"].max())
print("Number of crop years:", olive_oil["Haverst period"].nunique())

First crop year: 1990/91
Last crop year: 2024/25
Number of crop years: 35


In [9]:
target_countries = ["Spain", "Türkiye"]

coverage = (
    olive_oil[olive_oil["Country"].isin(target_countries)]
    .groupby(["Country", "Indicator"])["Haverst period"]
    .agg(["min", "max", "count"])
)

coverage

min      max  count
Country Indicator                         
Spain   C          1990/91  2024/25     35
        E          1990/91  2024/25     35
        I          1990/91  2024/25     35
        P          1990/91  2024/25     34
Türkiye C          1990/91  2024/25     25
        E          1990/91  2024/25     25
        I          1990/91  2024/25     25
        P          1990/91  2024/25     25

Spain is almost complete. Türkiye is missing a block of years in the dashboard file. We will fill that gap from the official IOC world balance sheets later.

## 5. Data Quality Checks

Now, we focus on Spain and Türkiye and check for duplicates, missing combinations, crop-year labels, and aggregate rows that should not be treated as countries.

In [10]:
target_df = olive_oil[olive_oil["Country"].isin(target_countries)].copy()

print("Rows:", target_df.shape[0])
target_df.head()

Rows: 239


,Haverst period,Country,Product Type,Indicator,Tonnes
137,1990/91,Spain,OO,C,394.1
138,1990/91,Spain,OO,E,65.8
139,1990/91,Spain,OO,I,26.7
140,1990/91,Spain,OO,P,639.4
156,1990/91,Türkiye,OO,C,55.0


In [11]:
duplicates = target_df.duplicated(
    subset=["Haverst period", "Country", "Indicator"]
).sum()

print("Duplicated country-year-indicator rows:", duplicates)

Duplicated country-year-indicator rows: 0


In [12]:
print("Crop years in the raw file:")
print(sorted(olive_oil["Haverst period"].unique()))

Crop years in the raw file:
['1990/91', '1991/92', '1992/93', '1993/94', '1994/95', '1995/96', '1996/97', '1997/98', '1998/99', '1999/00', '2000/01', '2001/02', '2002/03', '2003/04', '2004/05', '2005/06', '2006/07', '2007/08', '2008/9', '2009/10', '2010/11', '2011/12', '2012/13', '2013/14', '2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21', '2021/22', '2022/23', '2023/24', '2024/25']


Some crop years are written as `1998/9` or `2000/1` instead of `1998/99` and `2000/01`. These need to be standardized before we can join years or fill gaps.

In [13]:
def normalize_crop_year(value):
    start_year = int(str(value).split("/")[0])
    end_year = start_year + 1
    return f"{start_year}/{str(end_year)[-2:].zfill(2)}"


target_df["crop_year"] = target_df["Haverst period"].map(normalize_crop_year)

print("Spain crop years:", target_df[target_df["Country"] == "Spain"]["crop_year"].nunique())
print("Türkiye crop years:", target_df[target_df["Country"] == "Türkiye"]["crop_year"].nunique())

Spain crop years: 35
Türkiye crop years: 25


In [14]:
spain_years = set(
    target_df.loc[target_df["Country"] == "Spain", "crop_year"]
)
turkiye_years = set(
    target_df.loc[target_df["Country"] == "Türkiye", "crop_year"]
)

print("Years in Spain but not Türkiye:")
print(sorted(spain_years - turkiye_years))
print("\nYears in Türkiye but not Spain:")
print(sorted(turkiye_years - spain_years))

Years in Spain but not Türkiye:
['1998/99', '1999/00', '2000/01', '2001/02', '2002/03', '2003/04', '2004/05', '2005/06', '2006/07', '2007/08']

Years in Türkiye but not Spain:
[]


In [15]:
spain_wide = (
    target_df[target_df["Country"] == "Spain"]
    .pivot_table(
        index="crop_year",
        columns="Indicator",
        values="Tonnes",
        aggfunc="first",
    )
)

spain_wide[spain_wide.isna().any(axis=1)]

Indicator,C,E,I,P
crop_year,,,,
2007/08,546.3,133.9,40.3,NaN


Türkiye is missing 1998/99 through 2007/08 in the dashboard extract. Spain is missing production for 2007/08. Both gaps exist in the official world balance sheets, so we fill them there instead of dropping the years.

In [16]:
print("Aggregate or residual names in the olive oil file:")
for country in sorted(olive_oil["Country"].unique()):
    if country in {
        "EU",
        "Other pr.coun.",
        "Oth.non-prod.",
        "E.B.L.U.",
    }:
        print("-", country)

Aggregate or residual names in the olive oil file:
- E.B.L.U.
- EU
- Oth.non-prod.
- Other pr.coun.


`EU` is the sum of EU members, including Spain. If we add Spain and EU together, Spain is counted twice. `Other pr.coun.` and `Oth.non-prod.` are residual buckets, not countries.

## 6. Data Cleaning

The reproducible cleaning lives in `src/preprocessing/preprocess_ioc.py`. It:

- keeps olive oil only
- standardizes crop-year labels
- converts thousand tonnes to tonnes
- fills the Türkiye gap and Spain's 2007/08 production from the world balances workbook
- adds world totals
- saves all entities and the Spain/Türkiye file

We run that script here so the notebook and the processed files stay in sync.

In [17]:
sys.path.insert(0, str(project_root / "src" / "preprocessing"))

from preprocess_ioc import preprocess_ioc

preprocess_ioc()

IOC preprocessing completed.
All entities: (1626, 9)
Spain and Türkiye: (70, 9)
Cells filled from world balances: 181
Mean implied stock change, Spain/Türkiye: 208,040 tonnes


## 7. Validate the Clean Data

Finally, we check that the cleaned Spain and Türkiye dataset has the expected structure.

In [18]:
clean_path = project_root / "data" / "processed" / "ioc_clean.csv"
clean_df = pd.read_csv(clean_path)

print("Rows:", clean_df.shape[0])
print("Columns:", clean_df.shape[1])
print(
    "Duplicated country-crop-year rows:",
    clean_df.duplicated(subset=["country", "crop_year"]).sum(),
)

clean_df.head()

Rows: 70
Columns: 9
Duplicated country-crop-year rows: 0


,country,crop_year,crop_year_start,entity_type,year_status,production_tonnes,consumption_tonnes,exports_tonnes,imports_tonnes
0,Spain,1990/91,1990,country,reported,639400.0,394100.0,65800.0,26700.0
1,Spain,1991/92,1991,country,reported,593000.0,418700.0,62800.0,31000.0
2,Spain,1992/93,1992,country,reported,623100.0,421400.0,51600.0,13100.0
3,Spain,1993/94,1993,country,reported,550900.0,421000.0,54600.0,54000.0
4,Spain,1994/95,1994,country,reported,538800.0,420000.0,54000.0,61600.0


In [19]:
clean_df.groupby("country")["crop_year"].agg(["min", "max", "count"])

,min,max,count
country,,,
Spain,1990/91,2024/25,35
Türkiye,1990/91,2024/25,35


In [20]:
clean_df.isna().sum()

country               0
crop_year             0
crop_year_start       0
entity_type           0
year_status           0
production_tonnes     0
consumption_tonnes    0
exports_tonnes        0
imports_tonnes        0
dtype: int64

In [21]:
(
    clean_df[clean_df["country"] == "Türkiye"]
    .sort_values("crop_year_start")
    .loc[
        lambda d: d["crop_year_start"].between(1997, 2008),
        [
            "crop_year",
            "production_tonnes",
            "consumption_tonnes",
            "exports_tonnes",
            "imports_tonnes",
        ],
    ]
)

,crop_year,production_tonnes,consumption_tonnes,exports_tonnes,imports_tonnes
42,1997/98,40000.0,85500.0,35000.0,0.0
43,1998/99,170000.0,85000.0,86000.0,1000.0
44,1999/00,70000.0,60000.0,16500.0,2000.0
45,2000/01,175000.0,72500.0,92000.0,0.0
46,2001/02,65000.0,55000.0,28000.0,0.0
47,2002/03,140000.0,50000.0,74000.0,0.0
48,2003/04,79000.0,46000.0,46000.0,0.0
49,2004/05,145000.0,60000.0,93500.0,0.0
50,2005/06,112000.0,50000.0,73000.0,0.0
51,2006/07,165000.0,80000.0,45000.0,0.0


The filled Türkiye years show the classic on/off harvest pattern. That is expected for olives and should not be smoothed away in later feature engineering.

In [22]:
all_path = project_root / "data" / "processed" / "ioc_all_countries.csv"
all_df = pd.read_csv(all_path)

print(all_df["entity_type"].value_counts())
print("\nWorld 2023/24 and 2024/25 production:")
print(
    all_df[
        (all_df["country"] == "World")
        & (all_df["crop_year"].isin(["2023/24", "2024/25"]))
    ][["crop_year", "production_tonnes", "consumption_tonnes"]]
)

entity_type
country      1506
aggregate      85
world          35
Name: count, dtype: int64

World 2023/24 and 2024/25 production:
     crop_year  production_tonnes  consumption_tonnes
1624   2023/24          2589000.0           2788500.0
1625   2024/25          3572000.0           3215000.0


## 8. Notes for later

- IOC crop years (October–September) will not line up 1:1 with FAOSTAT calendar years.
- 2024/25 is still provisional.
- Spain exports in this file include shipments inside the EU. That is the right series for a Spain vs Türkiye market comparison.
- World production in 2023/24 is 2,589,000 tonnes and 2024/25 is 3,572,000 tonnes, matching the IOC December 2025 market update.
